In [1]:
from os import chmod

import pandas as pd
import numpy as np
import networkx as nx
from func.UsefulFunctions import *

In [2]:
#Start with the data for artists

artistDF = pd.read_parquet("Data/Network_artistsWikidata.parquet")


We will be linking artists to their influences, record labels, genres.

In [3]:
artistDF.head()

,artist,occupation,gender,occupationLabel,genderLabel,countryCitizenship,influencedBY,influencedBYLabel,statementCount,genre,article,musicBrainzID,recordLabel,recordLabelLabel,countryCitizenshipLabel,genreLabel,artistLabel
0,Q3176190,Q639669,Q6581072,musician,female,Q30,NaN,NaN,NaN,Q43343,https://en.wikipedia.org/wiki/Linda_Gail_Lewis,a2fe484f-4f35-4e0e-a954-2f6d5843b009,Q7544232,Smash Records,United States,folk music,Linda Gail Lewis
1,Q8065552,Q183945,Q6581097,record producer,male,Q145,NaN,NaN,NaN,NaN,https://en.wikipedia.org/wiki/Zalon,19c43212-18cf-4f02-baff-b794de3cdf02,NaN,NaN,United Kingdom,NaN,Zalon
2,Q191819,Q806349,Q6581097,bandleader,male,Q174193,NaN,NaN,NaN,Q9730,https://en.wikipedia.org/wiki/George_Martin,26fa8b67-6c7f-406c-ad64-a1d070092df2,Q21077,Warner Music Group,United Kingdom of Great Britain and Ireland,classical music,George Martin
3,Q21934026,Q183945,Q6581097,record producer,male,NaN,NaN,NaN,NaN,Q37073,https://en.wikipedia.org/wiki/Andrew_Scheps,ee11737c-d50c-4b04-ae5f-56e1f6373116,NaN,NaN,NaN,pop music,Andrew Scheps
4,Q1005957,Q55960555,Q6581097,recording artist,male,Q28,NaN,NaN,NaN,NaN,NaN,6054baf8-3a47-4c3a-8c51-eb976f69ecfc,NaN,NaN,Hungary,NaN,László Földes


In [4]:
# artistDF["Node_Name"] = artistDF["artist"] + "-" +artistDF["artistLabel"]
# artistDF["Genre_Name"] = artistDF["genre"] + "-" +artistDF["genreLabel"]
# artistDF["recordLabel_Name"] = artistDF["recordLabel"] + "-" +artistDF["recordLabelLabel"]
# artistDF["Influence_Name"] = artistDF["influencedBY"] + "-" +artistDF["influencedBYLabel"]


In [5]:
graphInfluence = addToNetwork4(artistDF, "influencedBY", "artist", "has influenced", "Musical Artist", "Musical artist","influencedBYLabel","artistLabel")


In [6]:
graphRecordLabel = addToNetwork4(artistDF, "recordLabel", "artist", "has/had artist","Record Label", "Musical Artist", "recordLabelLabel", "artistLabel")

In [7]:
graphGenre = addToNetwork4(artistDF, "genre", "artist", "has been participated in by", "Genre", "Musical Artist", "genreLabel", "artistLabel")

In [8]:
#compose all the graphs
firstGraph = nx.compose(graphInfluence, graphRecordLabel)
artistGraph = nx.compose(firstGraph, graphGenre)


In [9]:
#See the nodes in artistGraph
artistGraph.nodes["Q122971758"]

{'EntityType': 'Musical Artist', 'Label': 'Ikkimel'}

In [10]:
nx.write_graphml(artistGraph, "./graphs/artistGraph.graphml")

In [11]:
artistGraph.to_undirected().edges("Q5084390")

EdgeDataView([('Q5084390', 'Q42455'), ('Q5084390', 'Q11975'), ('Q5084390', 'Q21077'), ('Q5084390', 'Q202440'), ('Q5084390', 'Q1998195'), ('Q5084390', 'Q2690060'), ('Q5084390', 'Q37073'), ('Q5084390', 'Q187760'), ('Q5084390', 'Q842324'), ('Q5084390', 'Q188450'), ('Q5084390', 'Q487914'), ('Q5084390', 'Q211756'), ('Q5084390', 'Q1298934'), ('Q5084390', 'Q104695865'), ('Q5084390', 'Q9162904')])

In [12]:
ikkimelEgo = nx.ego_graph(artistGraph, "Q122971758", radius=3, center=True, undirected = True).copy()
print(ikkimelEgo.number_of_nodes())

39


In [13]:
sufjanEgo = nx.ego_graph(artistGraph, "Q319502", radius=3, center=True, undirected = True).copy()
print(sufjanEgo.number_of_nodes())
nx.write_graphml(sufjanEgo, "./graphs/sufjanEgo.graphml")

15549


In [14]:
#Subgraph of bjork, fka twigs, electronic music, hip-hop, arca
twigsEgoGraph = nx.ego_graph(artistGraph, "Q15291033", radius=3, center=True, undirected = True).copy()
#electronicEgoGraph = nx.ego_graph(artistGraph, "Q9778", radius=2, center=True,undirected = True).copy()
#subgraph = nx.compose(electronicEgoGraph, bjorkEgoGraph)
print(len(twigsEgoGraph.nodes()))
nx.write_graphml(twigsEgoGraph, "./graphs/twigsEgoGraph.graphml")
nx.write_graphml(ikkimelEgo, "./graphs/ikkimelEgo.graphml")

591
